# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> Self-contained (rebuilds features from raw parquet like the earlier notebooks) so it runs standalone. Same target proxy, same mid-panel month, same exclusions as w03/w04.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Random Forest classifier**, from the session's menu, chosen for three reasons specific to Lane 3 (Structured Content Archetype Clustering):

1. It naturally combines heterogeneous inputs — numeric performance signals, one-hot categorical content attributes, and a KMeans **archetype cluster label** — without me hand-engineering interactions between them.
2. It ties the lane's actual work (clustering) into the modeling week directly: I fit KMeans on the content-month feature space first, then feed each item's cluster membership into the Random Forest as an input feature. The question this answers is genuinely lane-specific — *does knowing a content item's behavioral archetype help predict whether it's underperforming, beyond the raw metrics alone?*
3. Permutation importance (also on the session's menu) gives an honest read on whether the cluster feature actually earns its place, rather than assuming clustering was useful because it's the capstone lane.

**Target (proxy label, same discipline as w03's leakage trap)**: `high_performer = avg_ctr > median(avg_ctr)`, thresholded on this month's own data — the same proxy used in the earlier notebooks, so the comparison against the baseline is apples-to-apples. `avg_ctr` itself is excluded from the feature set, since it's what defines the label (that was exactly the leakage lesson from w03).

In [1]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MID_MONTH = "2026-03"

raw = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS avg_ctr,
        AVG(f.gsc_avg_position) AS avg_position,
        SUM(f.gsc_impressions) AS total_impressions,
        SUM(f.ga4_sessions) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS sessions_per_impression,
        SUM(f.ga4_engaged_sessions) * 1.0 / NULLIF(SUM(f.ga4_sessions), 0) AS engagement_rate,
        SUM(f.sessions_ai) * 1.0 / NULLIF(SUM(f.ga4_sessions), 0) AS ai_referral_share,
        DATE_DIFF('day', MAX(d.content_created_date), DATE '{MID_MONTH}-01') AS content_age_days,
        MAX(d.word_count) AS word_count,
        MAX(d.search_volume) AS search_volume,
        MAX(d.competition) AS competition,
        MAX(d.content_type) AS content_type,
        MAX(d.main_intent) AS main_intent
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet') f
    JOIN read_parquet('{REL}/dim_content.parquet') d ON f.content_hash_id = d.content_hash_id
    JOIN read_parquet('{REL}/dim_clients.parquet') c ON f.client_hash_id = c.client_hash_id
    WHERE c.access_profile = 'gsc_and_ga4'
      AND d.is_deleted IS FALSE
      AND d.is_published IS TRUE
    GROUP BY 1, 2
""").df()
raw = raw.dropna(subset=['avg_ctr', 'avg_position'])

ratio_cols = ['sessions_per_impression', 'engagement_rate', 'ai_referral_share']
raw[ratio_cols] = raw[ratio_cols].fillna(0)
raw['word_count'] = raw['word_count'].fillna(raw['word_count'].median())
raw['search_volume'] = raw['search_volume'].fillna(0)
raw['competition'] = raw['competition'].fillna(raw['competition'].median())
raw['content_type'] = raw['content_type'].fillna('unknown')
raw['main_intent'] = raw['main_intent'].fillna('unknown')

raw['high_performer'] = (raw['avg_ctr'] > raw['avg_ctr'].median()).astype(int)

# --- Lane-3 tie-in: fit KMeans (k=6, matching the ML-03 clustering pass) on scaled numeric
# signals EXCLUDING avg_ctr (the label source), then attach cluster membership as a feature ---
cluster_input_cols = ['avg_position', 'total_impressions', 'sessions_per_impression',
                       'engagement_rate', 'ai_referral_share', 'content_age_days',
                       'word_count', 'search_volume', 'competition']
scaler = StandardScaler()
X_cluster = scaler.fit_transform(raw[cluster_input_cols])
kmeans = KMeans(n_clusters=6, random_state=0, n_init=10)
raw['archetype_cluster'] = kmeans.fit_predict(X_cluster).astype(str)

print(f"Rows: {len(raw)}")
print(f"High performer rate: {raw['high_performer'].mean():.1%}")
print(f"\nArchetype cluster sizes:\n{raw['archetype_cluster'].value_counts().sort_index()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 129305
High performer rate: 40.9%

Archetype cluster sizes:
archetype_cluster
0    50763
1      449
2    15527
3     1405
4    61114
5       47
Name: count, dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_hash_id`, not time-aware.** All content items in this notebook come from the same mid-panel month, so there's no future/past split to make honestly here (that's what the sealed final month is for, later). What matters is grouping: content items from the same client tend to share editorial style, template, and audience — if the same client's items land in both train and test, the model could partly "memorize" that client's baseline behavior rather than learn a generalizable archetype pattern. `GroupShuffleSplit` on `client_hash_id` guarantees no client appears in both sides, so the evaluation reflects generalization to unseen clients, which is the actual question Lane 3 cares about.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
train_idx, test_idx = next(gss.split(raw, groups=raw['client_hash_id']))

train_df = raw.iloc[train_idx].reset_index(drop=True)
test_df = raw.iloc[test_idx].reset_index(drop=True)

overlap = set(train_df['client_hash_id']) & set(test_df['client_hash_id'])
print(f"Train rows: {len(train_df)}  ({train_df['client_hash_id'].nunique()} clients)")
print(f"Test rows:  {len(test_df)}  ({test_df['client_hash_id'].nunique()} clients)")
print(f"Client overlap between train and test: {len(overlap)}  <- should be 0")

Train rows: 96859  (25 clients)
Test rows:  32446  (12 clients)
Client overlap between train and test: 0  <- should be 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Baseline metric, re-established here for a direct comparison: the Week-4 rule's score (`expected_ctr_for_position_bucket − actual_ctr`) used as a single ranking signal to predict `high_performer`, evaluated by AUC on this same grouped test split. The model below gets the full engineered feature set — including `archetype_cluster` — evaluated the identical way.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# --- Baseline: recompute the Week-4 rule's score on the SAME split, evaluate as a single-signal predictor ---
position_bins = [0, 3, 10, 20, 50, float('inf')]
position_labels = ['1-3', '4-10', '11-20', '21-50', '51+']
train_df['position_bucket'] = pd.cut(train_df['avg_position'], bins=position_bins, labels=position_labels)
test_df['position_bucket'] = pd.cut(test_df['avg_position'], bins=position_bins, labels=position_labels)

benchmark = train_df.groupby('position_bucket', observed=True)['avg_ctr'].mean().rename('expected_ctr_bucket')
test_scored = test_df.merge(benchmark, on='position_bucket', how='left')
test_scored['baseline_score'] = test_scored['expected_ctr_bucket'] - test_scored['avg_ctr']
test_scored = test_scored.dropna(subset=['baseline_score'])

baseline_auc = roc_auc_score(test_scored['high_performer'], -test_scored['baseline_score'])

# --- Model: Random Forest on the full feature set, including archetype_cluster ---
feature_cols_numeric = ['avg_position', 'total_impressions', 'sessions_per_impression',
                         'engagement_rate', 'ai_referral_share', 'content_age_days',
                         'word_count', 'search_volume', 'competition']
cat_cols = ['content_type', 'main_intent', 'archetype_cluster']

train_enc = pd.get_dummies(train_df, columns=cat_cols, prefix=['ctype', 'intent', 'cluster'])
test_enc = pd.get_dummies(test_df, columns=cat_cols, prefix=['ctype', 'intent', 'cluster'])
test_enc = test_enc.reindex(columns=train_enc.columns, fill_value=0)  # align columns exactly

feature_cols_model = [c for c in train_enc.columns
                       if c not in ('content_hash_id', 'client_hash_id', 'avg_ctr', 'high_performer', 'position_bucket')]

rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=0, class_weight='balanced')
rf.fit(train_enc[feature_cols_model], train_enc['high_performer'])
model_auc = roc_auc_score(test_enc['high_performer'], rf.predict_proba(test_enc[feature_cols_model])[:, 1])

comparison = pd.DataFrame({
    'method': ['Week-4 baseline rule (single signal)', 'Week-5 Random Forest (full features + archetype cluster)'],
    'auc': [baseline_auc, model_auc]
})
print(comparison.to_string(index=False))

                                                  method      auc
                    Week-4 baseline rule (single signal) 0.962770
Week-5 Random Forest (full features + archetype cluster) 0.927109


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import confusion_matrix

# --- Permutation importance: what does the model actually lean on? ---
perm = permutation_importance(rf, test_enc[feature_cols_model], test_enc['high_performer'],
                                n_repeats=15, random_state=0, scoring='roc_auc')
importance_df = pd.DataFrame({
    'feature': feature_cols_model,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)
print("Top 10 features by permutation importance:")
print(importance_df.head(10).to_string(index=False))

cluster_features = [f for f in feature_cols_model if f.startswith('cluster_')]
cluster_importance = importance_df[importance_df['feature'].isin(cluster_features)]['importance_mean'].sum()
print(f"\nCombined importance of all archetype_cluster dummy features: {cluster_importance:.4f}")
print("(This is the honest check on Section 1's claim — if this is near zero, the clustering")
print("work didn't actually help the model beyond what the raw metrics already captured.)")

# --- Error analysis ---
test_enc['predicted'] = rf.predict(test_enc[feature_cols_model])
cm = confusion_matrix(test_enc['high_performer'], test_enc['predicted'])
print(f"\nConfusion matrix:\n{cm}")

errors = test_enc[test_enc['high_performer'] != test_enc['predicted']]
print(f"\nMisclassified: {len(errors)} of {len(test_enc)} ({len(errors)/len(test_enc):.1%})")
print("\nMisclassified rows by archetype cluster:")
print(test_df.merge(errors[['content_hash_id']], on='content_hash_id')['archetype_cluster'].value_counts())
print("\nA cluster that's over-represented in errors relative to its overall size suggests that")
print("archetype isn't behaving consistently — its members don't share a clean high/low-CTR pattern,")
print("which is a useful, honest limitation to name rather than paper over with a higher overall AUC.")

Top 10 features by permutation importance:
                feature  importance_mean  importance_std
      total_impressions         0.153126        0.002019
sessions_per_impression         0.073506        0.001475
        engagement_rate         0.007900        0.000263
           avg_position         0.004409        0.000166
  ctype_keyword article         0.001754        0.000226
       content_age_days         0.000814        0.000219
         intent_unknown         0.000720        0.000170
              cluster_4         0.000610        0.000127
              cluster_0         0.000380        0.000136
             word_count         0.000345        0.000241

Combined importance of all archetype_cluster dummy features: 0.0013
(This is the honest check on Section 1's claim — if this is near zero, the clustering
work didn't actually help the model beyond what the raw metrics already captured.)

Confusion matrix:
[[19784  1308]
 [ 3159  8195]]

Misclassified: 4467 of 32446 (13.8%)

Mis

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.